# 03 · Least squares — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1–2. Noise and station count

Increasing noise broadens the excited singular modes. Removing stations lowers
rank and leaves more slip combinations unconstrained.

## 3. Whitening

For diagonal covariance, divide each row and datum by its sigma. `lstsq` on the
whitened system matches weighted least squares.

## 4. Small singular vectors

The final right singular vector is the model pattern least visible to the data;
large unregularized oscillations align with this subspace.


In [ ]:
result = geodef.solve(fault, datasets=gnss, components="dip")
G = geodef.greens.matrix(fault, gnss)[:, fault.n_patches:]
weights = 1.0 / gnss.sigma
manual, *_ = np.linalg.lstsq(G * weights[:, None], gnss.obs * weights, rcond=None)
assert np.allclose(manual, result.dip_slip)
_, singular_values, Vt = np.linalg.svd(G * weights[:, None], full_matrices=False)
alignment = abs(np.dot(result.dip_slip, Vt[-1])) / np.linalg.norm(result.dip_slip)
print("reduced chi2:", result.reduced_chi2)
print("condition number:", singular_values[0] / singular_values[-1])
print("weakest-mode alignment:", alignment)


## Interpretation and alternatives

An excellent weighted fit can coexist with strong projection onto weak model modes.
